# Gold Standard Preparation

This notebook prepares the gold standard validation dataset from public domain book excerpts.

Each excerpt has a known target reading age. We calculate readability scores and expand the dataset
across target ages and model versions, ready for evaluation and optional GPT simplification.

**Source texts:** Public domain fiction (Beatrix Potter, Lewis Carroll, L. Frank Baum, R.L. Stevenson,
Mark Twain, Charles Dickens, Charlotte Brontë, Jane Austen)

**Output:** `gold_standard.csv`

In [1]:
import pandas as pd
import re
import ssl
import nltk
import textstat
from nltk.corpus import stopwords

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/alanabarrett-
[nltk_data]     frew/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Load source excerpts

In [2]:
df = pd.read_csv('book_excerpts.csv')
print(f"Loaded {len(df)} excerpts")
df[['id', 'title', 'author', 'target_age']]

Loaded 8 excerpts


,id,title,author,target_age
0,1,The Tale of Peter Rabbit,Beatrix Potter,6
1,2,Alice's Adventures in Wonderland,Lewis Carroll,8
2,3,The Wonderful Wizard of Oz,L. Frank Baum,9
3,4,The Adventures of Tom Sawyer,Mark Twain,10
4,5,Treasure Island,Robert Louis Stevenson,12
5,6,Great Expectations,Charles Dickens,13
6,7,Jane Eyre,Charlotte Bronte,15
7,8,Pride and Prejudice,Jane Austen,17


## Text preprocessing

In [3]:
stop = stopwords.words('english')

def tokenize(text):
    cleaned = re.sub('[^A-Za-z]', ' ', text.lower())
    return [w for w in cleaned.split() if w not in stop and len(w) > 1]

df['tokenized_text'] = df['text_content'].apply(tokenize)
df['clean_text'] = df['tokenized_text'].str.join(' ')
df['token_count'] = df['clean_text'].apply(lambda x: len(x.split()))

df[['title', 'token_count']].head(10)

,title,token_count
0,The Tale of Peter Rabbit,79
1,Alice's Adventures in Wonderland,69
2,The Wonderful Wizard of Oz,86
3,The Adventures of Tom Sawyer,74
4,Treasure Island,81
5,Great Expectations,76
6,Jane Eyre,69
7,Pride and Prejudice,63


## Readability scoring

In [4]:
df['flesch_kincaid_score_original'] = df['text_content'].apply(textstat.flesch_kincaid_grade)

# Convert FK grade to approximate UK reading age (FK grade + 5 gives rough UK reading age)
df['reading_age_original'] = df['flesch_kincaid_score_original'].apply(
    lambda x: round(x + 5) if pd.notnull(x) else None
)

df[['title', 'target_age', 'flesch_kincaid_score_original', 'reading_age_original']]

,title,target_age,flesch_kincaid_score_original,reading_age_original
0,The Tale of Peter Rabbit,6,6.215173,11
1,Alice's Adventures in Wonderland,8,12.977988,18
2,The Wonderful Wizard of Oz,9,8.513492,14
3,The Adventures of Tom Sawyer,10,8.258261,13
4,Treasure Island,12,14.945405,20
5,Great Expectations,13,9.263106,14
6,Jane Eyre,15,13.096167,18
7,Pride and Prejudice,17,7.143205,12


## Expand for target ages and models

Each excerpt is paired with multiple target reading ages and model versions,
creating rows for the simplification evaluation pipeline.

In [5]:
target_ages = [6, 8, 9, 11, 13, 15, 17]
models = ['gpt-3.5-turbo', 'gpt-4o-mini', 'gpt-5-nano', 'gpt-5', 'gpt-6-luna']

new_rows = []
for _, row in df.iterrows():
    for target_age in target_ages:
        for model in models:
            new_rows.append({
                'id': row['id'],
                'title': row['title'],
                'author': row['author'],
                'text_content': row['text_content'],
                'tokenized_text': row['tokenized_text'],
                'clean_text': row['clean_text'],
                'token_count': row['token_count'],
                'flesch_kincaid_score_original': row['flesch_kincaid_score_original'],
                'reading_age_original': row['reading_age_original'],
                'source_target_age': row['target_age'],
                'model': model,
                'target_age': target_age,
                'simplified_text': '',
                'fk_score_simplified': '',
                'reading_age_simplified': '',
                'token_count_simplified': '',
            })

gold_df = pd.DataFrame(new_rows)
print(f"Expanded to {len(gold_df)} rows ({len(df)} excerpts × {len(target_ages)} target ages × {len(models)} models)")
gold_df.head()

Expanded to 280 rows (8 excerpts × 7 target ages × 5 models)


,id,title,author,text_content,tokenized_text,clean_text,token_count,flesch_kincaid_score_original,reading_age_original,source_target_age,model,target_age,simplified_text,fk_score_simplified,reading_age_simplified,token_count_simplified
0,1,The Tale of Peter Rabbit,Beatrix Potter,Once upon a time there were four little Rabbit...,"[upon, time, four, little, rabbits, names, flo...",upon time four little rabbits names flopsy mop...,79,6.215173,11,6,gpt-3.5-turbo,6,,,,
1,1,The Tale of Peter Rabbit,Beatrix Potter,Once upon a time there were four little Rabbit...,"[upon, time, four, little, rabbits, names, flo...",upon time four little rabbits names flopsy mop...,79,6.215173,11,6,gpt-4o-mini,6,,,,
2,1,The Tale of Peter Rabbit,Beatrix Potter,Once upon a time there were four little Rabbit...,"[upon, time, four, little, rabbits, names, flo...",upon time four little rabbits names flopsy mop...,79,6.215173,11,6,gpt-5-nano,6,,,,
3,1,The Tale of Peter Rabbit,Beatrix Potter,Once upon a time there were four little Rabbit...,"[upon, time, four, little, rabbits, names, flo...",upon time four little rabbits names flopsy mop...,79,6.215173,11,6,gpt-5,6,,,,
4,1,The Tale of Peter Rabbit,Beatrix Potter,Once upon a time there were four little Rabbit...,"[upon, time, four, little, rabbits, names, flo...",upon time four little rabbits names flopsy mop...,79,6.215173,11,6,gpt-6-luna,6,,,,


## Save gold standard

In [6]:
gold_df.to_csv('gold_standard.csv', index=False)
print(f"Saved gold_standard.csv with {len(gold_df)} rows")
gold_df[['title', 'author', 'target_age', 'model', 'reading_age_original']].head(20)

Saved gold_standard.csv with 280 rows


,title,author,target_age,model,reading_age_original
0,The Tale of Peter Rabbit,Beatrix Potter,6,gpt-3.5-turbo,11
1,The Tale of Peter Rabbit,Beatrix Potter,6,gpt-4o-mini,11
2,The Tale of Peter Rabbit,Beatrix Potter,6,gpt-5-nano,11
3,The Tale of Peter Rabbit,Beatrix Potter,6,gpt-5,11
4,The Tale of Peter Rabbit,Beatrix Potter,6,gpt-6-luna,11
5,The Tale of Peter Rabbit,Beatrix Potter,8,gpt-3.5-turbo,11
6,The Tale of Peter Rabbit,Beatrix Potter,8,gpt-4o-mini,11
7,The Tale of Peter Rabbit,Beatrix Potter,8,gpt-5-nano,11
8,The Tale of Peter Rabbit,Beatrix Potter,8,gpt-5,11
9,The Tale of Peter Rabbit,Beatrix Potter,8,gpt-6-luna,11
